In [1]:
pip install xgboost


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# MODEL: XGBOOST
# 5-FOLD CROSS VALIDATION
# PHISHING URL DETECTION
# ============================================================


# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# STEP 2: LOAD CLEANED DATASET
# ============================================================

print("=" * 70)
print("XGBOOST - 5-FOLD CROSS VALIDATION")
print("PHISHING URL DETECTION")
print("=" * 70)

df = pd.read_csv('phishing_website_cleaned.csv')

print("\nDataset shape:")
print(df.shape)


# ============================================================
# STEP 3: DEFINE URL FEATURES
# SAME FEATURES AS BASELINE XGBOOST MODEL
# ============================================================

URL_FEATURES = [

    # --------------------------------------------------------
    # URL CHARACTER FEATURES
    # --------------------------------------------------------

    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    # --------------------------------------------------------
    # DOMAIN CHARACTER FEATURES
    # --------------------------------------------------------

    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    # --------------------------------------------------------
    # DOMAIN STATISTICS
    # --------------------------------------------------------

    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',

    # --------------------------------------------------------
    # SECURITY / URL FEATURES
    # --------------------------------------------------------

    'tls_ssl_certificate',
    'url_shortened'
]


# ============================================================
# STEP 4: CHECK AVAILABLE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: FEATURE CHECK")
print("=" * 70)

available_features = [
    col for col in URL_FEATURES
    if col in df.columns
]

missing_features = [
    col for col in URL_FEATURES
    if col not in df.columns
]

print("\nTotal requested features:", len(URL_FEATURES))
print("Available features:", len(available_features))

if missing_features:

    print("\nMissing features:")
    for col in missing_features:
        print("-", col)

else:

    print("\n✓ All requested URL features are available.")


# ============================================================
# STEP 5: SEPARATE FEATURES AND TARGET
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: SEPARATE FEATURES AND TARGET")
print("=" * 70)

X = df[available_features].copy()
y = df['phishing'].copy()

print("\nFeatures shape:")
print(X.shape)

print("\nTarget shape:")
print(y.shape)

print("\nNumber of features:")
print(X.shape[1])


# ============================================================
# STEP 6: CHECK TARGET FOR MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: CHECK TARGET")
print("=" * 70)

print("\nMissing target values:", y.isna().sum())

if y.isna().sum() > 0:

    valid_rows = y.notna()

    X = X.loc[valid_rows].copy()
    y = y.loc[valid_rows].copy()

    print("\n✓ Rows with missing target values removed.")

else:

    print("\n✓ No missing target values found.")


# ============================================================
# STEP 7: REMOVE CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: CONSTANT FEATURE CHECK")
print("=" * 70)

constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

print("\nNumber of constant features:",
      len(constant_features))

if constant_features:

    print("\nRemoving constant features:")

    for col in constant_features:
        print("-", col)

    X = X.drop(
        columns=constant_features
    )

else:

    print("\n✓ No constant features found.")


print("\nFeatures after removing constant features:",
      X.shape[1])


# ============================================================
# STEP 8: CHECK MISSING VALUES IN FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: FEATURE MISSING VALUE CHECK")
print("=" * 70)

total_missing = X.isna().sum().sum()

print("\nTotal missing values in features:",
      total_missing)

if total_missing > 0:

    print("\nMissing values by feature:")

    print(
        X.isna()
        .sum()
        .loc[lambda x: x > 0]
    )

    # Fill remaining missing values with median
    X = X.fillna(X.median())

    print("\n✓ Missing feature values filled with median.")

else:

    print("\n✓ No missing feature values found.")


# ============================================================
# STEP 9: DISPLAY TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("STEP 6: TARGET DISTRIBUTION")
print("=" * 70)

print("\nTarget distribution:")

print(y.value_counts())

print("\nTarget percentage:")

print(
    y.value_counts(normalize=True) * 100
)


# ============================================================
# STEP 10: CREATE XGBOOST MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 7: CREATE XGBOOST MODEL")
print("=" * 70)

xgb_model = XGBClassifier(

    n_estimators=100,

    max_depth=6,

    learning_rate=0.1,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric='logloss',

    n_jobs=-1
)

print("\n✓ XGBoost Classifier created.")

print("\nModel Parameters:")
print("n_estimators      :", 100)
print("max_depth         :", 6)
print("learning_rate     :", 0.1)
print("subsample         :", 0.8)
print("colsample_bytree  :", 0.8)
print("random_state      :", 42)


# ============================================================
# STEP 11: CREATE PIPELINE
# ============================================================
# StandardScaler is included because your baseline model
# uses StandardScaler before XGBoost.
#
# The scaler is fitted separately inside each CV fold,
# preventing preprocessing leakage between folds.
# ============================================================

model_pipeline = Pipeline([

    ('scaler', StandardScaler()),

    ('xgboost', xgb_model)

])

print("\n✓ XGBoost pipeline created.")


# ============================================================
# STEP 12: CREATE 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 8: CREATE 5-FOLD CROSS VALIDATION")
print("=" * 70)

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

print("\nNumber of folds:", cv.n_splits)

print("Cross-validation method:",
      "StratifiedKFold")

print("Shuffle:", True)

print("Random state:", 42)

print("Sampling method: None")


# ============================================================
# STEP 13: DEFINE EVALUATION METRICS
# ============================================================

print("\n" + "=" * 70)
print("STEP 9: EVALUATION METRICS")
print("=" * 70)

scoring = {

    'accuracy': 'accuracy',

    'precision': 'precision',

    'recall': 'recall',

    'f1': 'f1'

}

print("\nMetrics:")

print("- Accuracy")

print("- Precision")

print("- Recall")

print("- F1-Score")


# ============================================================
# STEP 14: RUN 5-FOLD CROSS VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("STEP 10: RUNNING 5-FOLD CROSS VALIDATION")
print("=" * 70)

cv_results = cross_validate(

    model_pipeline,

    X,

    y,

    cv=cv,

    scoring=scoring,

    return_train_score=False,

    n_jobs=-1
)

print("\n✓ 5-Fold Cross Validation completed successfully.")


# ============================================================
# STEP 15: DISPLAY INDIVIDUAL FOLD RESULTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 70)

for i in range(5):

    print(f"\nFold {i + 1}")

    print("-" * 45)

    print(
        "Accuracy :",
        f"{cv_results['test_accuracy'][i] * 100:.2f}%"
    )

    print(
        "Precision:",
        f"{cv_results['test_precision'][i] * 100:.2f}%"
    )

    print(
        "Recall   :",
        f"{cv_results['test_recall'][i] * 100:.2f}%"
    )

    print(
        "F1-Score :",
        f"{cv_results['test_f1'][i] * 100:.2f}%"
    )


# ============================================================
# STEP 16: CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

accuracy_mean = (
    cv_results['test_accuracy'].mean()
)

accuracy_std = (
    cv_results['test_accuracy'].std()
)

precision_mean = (
    cv_results['test_precision'].mean()
)

precision_std = (
    cv_results['test_precision'].std()
)

recall_mean = (
    cv_results['test_recall'].mean()
)

recall_std = (
    cv_results['test_recall'].std()
)

f1_mean = (
    cv_results['test_f1'].mean()
)

f1_std = (
    cv_results['test_f1'].std()
)


# ============================================================
# STEP 17: CREATE RESULTS TABLE
# ============================================================

results_table = pd.DataFrame({

    'Fold': [1, 2, 3, 4, 5],

    'Accuracy':
        cv_results['test_accuracy'],

    'Precision':
        cv_results['test_precision'],

    'Recall':
        cv_results['test_recall'],

    'F1-Score':
        cv_results['test_f1']
})


# ============================================================
# STEP 18: DISPLAY RESULTS TABLE
# ============================================================

print("\n" + "=" * 70)
print("CROSS VALIDATION RESULTS TABLE")
print("=" * 70)

print(
    results_table.to_string(
        index=False
    )
)


# ============================================================
# STEP 19: DISPLAY MEAN ± STANDARD DEVIATION
# ============================================================

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION SUMMARY")
print("=" * 70)

print("\nXGBoost Performance")
print("-" * 50)

print(
    f"Accuracy : "
    f"{accuracy_mean * 100:.2f}% ± "
    f"{accuracy_std * 100:.2f}%"
)

print(
    f"Precision: "
    f"{precision_mean * 100:.2f}% ± "
    f"{precision_std * 100:.2f}%"
)

print(
    f"Recall   : "
    f"{recall_mean * 100:.2f}% ± "
    f"{recall_std * 100:.2f}%"
)

print(
    f"F1-Score : "
    f"{f1_mean * 100:.2f}% ± "
    f"{f1_std * 100:.2f}%"
)


# ============================================================
# STEP 20: TRAIN FINAL XGBOOST MODEL
# ============================================================

print("\n" + "=" * 70)
print("STEP 11: TRAIN FINAL MODEL")
print("=" * 70)

final_xgb_model = Pipeline([

    ('scaler', StandardScaler()),

    ('xgboost', XGBClassifier(

        n_estimators=100,

        max_depth=6,

        learning_rate=0.1,

        subsample=0.8,

        colsample_bytree=0.8,

        random_state=42,

        eval_metric='logloss',

        n_jobs=-1
    ))

])


final_xgb_model.fit(
    X,
    y
)

print("\n✓ Final XGBoost model trained successfully.")


# ============================================================
# STEP 21: FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("XGBOOST 5-FOLD CROSS VALIDATION COMPLETED")
print("=" * 70)

print("\nDataset:")
print("phishing_website_cleaned.csv")

print("\nNumber of samples:")
print(len(X))

print("\nNumber of features:")
print(X.shape[1])

print("\nCross Validation:")
print("5-Fold Stratified Cross Validation")

print("\nSampling:")
print("None")

print("\nModel:")
print("XGBoost Classifier")

print("\nFinal Cross-Validation Performance:")
print("-" * 50)

print(
    f"Accuracy : "
    f"{accuracy_mean * 100:.2f}% ± "
    f"{accuracy_std * 100:.2f}%"
)

print(
    f"Precision: "
    f"{precision_mean * 100:.2f}% ± "
    f"{precision_std * 100:.2f}%"
)

print(
    f"Recall   : "
    f"{recall_mean * 100:.2f}% ± "
    f"{recall_std * 100:.2f}%"
)

print(
    f"F1-Score : "
    f"{f1_mean * 100:.2f}% ± "
    f"{f1_std * 100:.2f}%"
)

print("\n✓ XGBoost 5-Fold Cross Validation is complete.")


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

In [4]:
# ============================================================
# MODEL 2: XGBOOST PHISHING URL DETECTION
# ============================================================

import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

df = pd.read_csv('phishing_website_cleaned.csv')

print("=" * 60)
print("XGBOOST PHISHING URL DETECTION")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

# ============================================================
# STEP 2: DEFINE EXACT SAME URL FEATURES AS RANDOM FOREST
# ============================================================

URL_FEATURES = [
    # URL character features
    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    # Domain character features
    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    # Domain statistics
    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',

    # Security / URL features
    'tls_ssl_certificate',
    'url_shortened'
]

# ============================================================
# STEP 3: CHECK AVAILABLE FEATURES
# ============================================================

available_features = [col for col in URL_FEATURES if col in df.columns]
missing_features = [col for col in URL_FEATURES if col not in df.columns]

print("\n" + "=" * 60)
print("FEATURE CHECK")
print("=" * 60)
print("\nNumber of available URL features:", len(available_features))

if missing_features:
    print("\nMissing features:", missing_features)

# ============================================================
# STEP 4: CREATE X AND y
# ============================================================

X = df[available_features].copy()
y = df['phishing'].copy()

print("\n" + "=" * 60)
print("FEATURE AND TARGET")
print("=" * 60)
print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [col for col in X.columns if X[col].nunique() <= 1]

print("\n" + "=" * 60)
print("CONSTANT FEATURE CHECK")
print("=" * 60)
print("\nNumber of constant features:", len(constant_features))

if constant_features:
    print("\nRemoving constant features:")
    for col in constant_features:
        print("-", col)
    X = X.drop(columns=constant_features)

print("\nFeatures after removing constant features:", X.shape[1])

# ============================================================
# STEP 6: TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)
print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

# ============================================================
# STEP 7: STANDARDIZATION
# ============================================================

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform ONLY test data
X_test_scaled = scaler.transform(X_test)

print("\n" + "=" * 60)
print("STANDARDIZATION")
print("=" * 60)
print("\n✓ Standardization completed")

# ============================================================
# STEP 8: XGBOOST MODEL TRAINING
# ============================================================

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

print("\n" + "=" * 60)
print("XGBOOST TRAINING")
print("=" * 60)
print("\n✓ XGBoost training completed successfully!")

# ============================================================
# STEP 9: PREDICTION ON TEST DATA
# ============================================================

y_pred = model.predict(X_test_scaled)

# ============================================================
# STEP 10: MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)
print(f"\nAccuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ============================================================
# STEP 11: EXTRACT FEATURES FROM NEW URL
# ============================================================

def extract_url_features(url):
    url = url.strip()

    if url.startswith('[') and '](' in url:
        url = url.split('](', 1)[1]
        if url.endswith(')'):
            url = url[:-1]

    url = url.strip()

    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    parsed = urlparse(url)
    full_url = url
    domain = parsed.netloc.split(':')[0]

    features = {}

    # URL CHARACTER FEATURES
    features['qty_dot_url'] = full_url.count('.')
    features['qty_hyphen_url'] = full_url.count('-')
    features['qty_underline_url'] = full_url.count('_')
    features['qty_slash_url'] = full_url.count('/')
    features['qty_questionmark_url'] = full_url.count('?')
    features['qty_equal_url'] = full_url.count('=')
    features['qty_at_url'] = full_url.count('@')
    features['qty_and_url'] = full_url.count('&')
    features['qty_exclamation_url'] = full_url.count('!')
    features['qty_space_url'] = full_url.count(' ')
    features['qty_tilde_url'] = full_url.count('~')
    features['qty_comma_url'] = full_url.count(',')
    features['qty_plus_url'] = full_url.count('+')
    features['qty_asterisk_url'] = full_url.count('*')
    features['qty_hashtag_url'] = full_url.count('#')
    features['qty_dollar_url'] = full_url.count('$')
    features['qty_percent_url'] = full_url.count('%')
    features['length_url'] = len(full_url)

    # DOMAIN CHARACTER FEATURES
    features['qty_dot_domain'] = domain.count('.')
    features['qty_hyphen_domain'] = domain.count('-')
    features['qty_underline_domain'] = domain.count('_')
    features['qty_slash_domain'] = domain.count('/')
    features['qty_questionmark_domain'] = domain.count('?')
    features['qty_equal_domain'] = domain.count('=')
    features['qty_at_domain'] = domain.count('@')
    features['qty_and_domain'] = domain.count('&')
    features['qty_exclamation_domain'] = domain.count('!')
    features['qty_space_domain'] = domain.count(' ')
    features['qty_tilde_domain'] = domain.count('~')
    features['qty_comma_domain'] = domain.count(',')
    features['qty_plus_domain'] = domain.count('+')
    features['qty_asterisk_domain'] = domain.count('*')
    features['qty_hashtag_domain'] = domain.count('#')
    features['qty_dollar_domain'] = domain.count('$')
    features['qty_percent_domain'] = domain.count('%')

    # DOMAIN STATISTICS
    features['qty_vowels_domain'] = sum(c.lower() in 'aeiou' for c in domain)
    features['domain_length'] = len(domain)

    # DOMAIN IN IP ADDRESS
    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
    features['domain_in_ip'] = int(bool(re.match(ip_pattern, domain)))

    # SERVER / CLIENT DOMAIN
    features['server_client_domain'] = int('server' in domain.lower() or 'client' in domain.lower())

    # EMAIL IN URL
    features['email_in_url'] = int('@' in full_url)

    # HTTPS / SSL
    features['tls_ssl_certificate'] = int(parsed.scheme == 'https')

    # URL SHORTENER
    shorteners = ['bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'is.gd', 'ow.ly']
    features['url_shortened'] = int(any(shortener in domain.lower() for shortener in shorteners))

    return features

# ============================================================
# STEP 12: PREPARE URL FOR MODEL
# ============================================================

def prepare_url_for_model(url):
    features = extract_url_features(url)
    url_df = pd.DataFrame([features])

    # Filter to only match X.columns and maintain the exact order
    url_df = url_df[X.columns]

    # Apply the same scaler
    url_scaled = scaler.transform(url_df)
    return url_scaled

# ============================================================
# STEP 13: PREDICT NEW URL
# ============================================================

def predict_url(url):
    url_scaled = prepare_url_for_model(url)

    prediction = model.predict(url_scaled)[0]
    probabilities = model.predict_proba(url_scaled)[0]

    legitimate_probability = probabilities[0]
    phishing_probability = probabilities[1]

    print("\n" + "=" * 55)
    print("XGBOOST URL PHISHING DETECTION")
    print("=" * 55)
    print("\nURL:")
    print(url)

    print(f"\nLegitimate probability: {legitimate_probability * 100:.2f}%")
    print(f"Phishing probability  : {phishing_probability * 100:.2f}%")

    if prediction == 1:
        print("\nResult: ⚠️ PHISHING URL")
    else:
        print("\nResult: ✅ LEGITIMATE URL")

    print("=" * 55)
    return prediction


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O